# RSNA 2022 PED: Drive zip -> GCS

这个笔记本用于：
- 挂载 Google Drive
- 认证 GCS
- 将 Drive 中的 zip 解压后上传到 GCS 指定目录

你只需要修改第一个代码单元的超参即可复用。

In [ ]:
# Cell 1: 超参配置（后续修改只改这里）
PROJECT_ID = "brats-preprocess"
GCS_BUCKET = "rsna_2022_ped_preprocessed_npy"

# 显式超参：zip 文件名 与 GCS 目标文件夹名
ZIP_FILE_NAME = "npy1.zip"
TARGET_GCS_FOLDER = "npy1"

# 你的 Drive 内 zip 所在路径
DRIVE_ZIP_PATH = f"/content/drive/MyDrive/dataset/pretrain/RSNA-2022-PED/{ZIP_FILE_NAME}"

# 上传目标前缀
GCS_TARGET_PREFIX = f"gs://{GCS_BUCKET}/{TARGET_GCS_FOLDER}"

print("Configuration:")
print(f"- PROJECT_ID: {PROJECT_ID}")
print(f"- GCS_BUCKET: {GCS_BUCKET}")
print(f"- ZIP_FILE_NAME: {ZIP_FILE_NAME}")
print(f"- TARGET_GCS_FOLDER: {TARGET_GCS_FOLDER}")
print(f"- DRIVE_ZIP_PATH: {DRIVE_ZIP_PATH}")
print(f"- GCS_TARGET_PREFIX: {GCS_TARGET_PREFIX}")

In [ ]:
# Cell 2: 挂载 Drive + 认证 GCS
from google.colab import drive, auth

drive.mount('/content/drive')
auth.authenticate_user()

import os
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID

!gcloud config set project {PROJECT_ID}
!gcloud config get-value project
!gsutil ls -L -b gs://{GCS_BUCKET}

In [ ]:
# Cell 3: 解压 zip 到临时目录
import os
import shutil
import zipfile
from pathlib import Path

if not os.path.exists(DRIVE_ZIP_PATH):
    raise FileNotFoundError(f"Zip not found: {DRIVE_ZIP_PATH}")

extract_root = Path('/content/tmp_rsna_extract')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zf:
    zf.extractall(extract_root)

entries = sorted([p for p in extract_root.iterdir()])
print(f"Extracted to: {extract_root}")
print(f"Top-level entries: {len(entries)}")
for p in entries[:10]:
    print(' -', p.name)

# 若 zip 内只有一个顶层目录，则上传该目录内容；否则上传整个解压目录内容
if len(entries) == 1 and entries[0].is_dir():
    upload_source = entries[0]
else:
    upload_source = extract_root

print(f"Upload source: {upload_source}")

In [ ]:
# Cell 4: 上传到 GCS 的 TARGET_GCS_FOLDER（例如 npy1）
import subprocess

# 确保目标前缀存在（创建占位对象后可在控制台立即看到文件夹）
subprocess.run(
    [
        'bash', '-lc',
        f"echo '' | gsutil cp - {GCS_TARGET_PREFIX}/.keep"
    ],
    check=True
)

# 上传解压后的内容到 gs://bucket/TARGET_GCS_FOLDER/
upload_cmd = f"gsutil -m cp -r '{upload_source}/.' '{GCS_TARGET_PREFIX}/'"
print('Running:', upload_cmd)
subprocess.run(['bash', '-lc', upload_cmd], check=True)

print('\nUpload done. Sample objects:')
subprocess.run(['bash', '-lc', f"gsutil ls '{GCS_TARGET_PREFIX}/' | head -20"], check=True)
subprocess.run(['bash', '-lc', f"gsutil ls -r '{GCS_TARGET_PREFIX}/**/*.npy' | head -20"], check=False)

In [ ]:
# Cell 5: 统计 npy 文件数量（可选）
!gsutil ls -r {GCS_TARGET_PREFIX}/**/*.npy | wc -l